# Model training for community crime classification

We train five classifiers -- Logistic Regression, Decision Tree, Random Forest,
Gradient Boosting, and XGBoost -- to predict community risk level
(Low / Medium / High). The pipeline uses functions from `src/model.py` for
data preparation and training, with added cross-validation and hyperparameter
tuning for the best-performing model.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

from src.data_loader import (
    load_or_fetch_crime_data, load_or_fetch_census_data,
    preprocess_crime_data, preprocess_census_data,
    create_community_features,
)
from src.model import (
    create_risk_labels, prepare_classification_data,
    train_classifiers, get_feature_importance, save_model,
)

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load engineered community data

In [ ]:
import os

engineered_path = '../data/community_features_engineered.csv'
if os.path.exists(engineered_path):
    community_df = pd.read_csv(engineered_path)
    print(f'Loaded engineered data: {len(community_df)} communities')
else:
    crime_raw = load_or_fetch_crime_data('../data')
    census_raw = load_or_fetch_census_data('../data')
    crime_df = preprocess_crime_data(crime_raw)
    census_df = preprocess_census_data(census_raw)
    community_df = create_community_features(crime_df, census_df)
    community_df = create_risk_labels(community_df)
    print(f'Built features from scratch: {len(community_df)} communities')

# Ensure risk labels exist
if 'risk_level' not in community_df.columns:
    community_df = create_risk_labels(community_df)

## 2. Prepare classification data

`prepare_classification_data` selects numeric feature columns, drops identifiers
and the target-related `total_crimes` column, fills NaNs with zero, and
label-encodes the risk level.

In [ ]:
X, y, label_encoder, feature_cols = prepare_classification_data(community_df)
print(f'Feature matrix: {X.shape}')
print(f'Classes: {list(label_encoder.classes_)}')
print(f'Class counts: {np.bincount(y)}')
print(f'Features: {feature_cols}')

## 3. Train all five classifiers

`train_classifiers` performs an 80/20 stratified split, scales features for
Logistic Regression, trains each model, and computes accuracy and F1 scores.

In [ ]:
trained_models, results, scaler, X_test, y_test = train_classifiers(X, y, random_state=42)

results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
results_df = results_df.round(4)
print(results_df.to_string())

## 4. Add XGBoost classifier

The base `train_classifiers` includes four models. We add XGBoost separately
and append its results to the comparison table.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

X_train, X_test_xgb, y_train, y_test_xgb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

xgb_model = XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1, use_label_encoder=False,
    eval_metric='mlogloss',
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test_xgb)

xgb_results = {
    'Accuracy': accuracy_score(y_test_xgb, y_pred_xgb),
    'F1 (Weighted)': f1_score(y_test_xgb, y_pred_xgb, average='weighted'),
    'F1 (Macro)': f1_score(y_test_xgb, y_pred_xgb, average='macro'),
}

trained_models['XGBoost'] = xgb_model
results['XGBoost'] = xgb_results

results_df = pd.DataFrame(results).T.round(4)
results_df.index.name = 'Model'
print(results_df.sort_values('Accuracy', ascending=False).to_string())

## 5. Cross-validation (5-fold)

With a small number of communities the single split can be noisy.
Cross-validation provides a more stable estimate.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SS

cv_models = {
    'Logistic Regression': Pipeline([('scaler', SS()), ('lr', LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'))]),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
}

cv_results = {}
for name, model in cv_models.items():
    acc_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    f1_scores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted')
    cv_results[name] = {
        'CV Accuracy': f'{acc_scores.mean():.3f} +/- {acc_scores.std():.3f}',
        'CV F1 (Weighted)': f'{f1_scores.mean():.3f} +/- {f1_scores.std():.3f}',
    }

cv_df = pd.DataFrame(cv_results).T
print(cv_df.to_string())

## 6. Hyperparameter tuning for the best model

We run a grid search on Gradient Boosting (typically the top performer on
this dataset) to fine-tune `n_estimators`, `max_depth`, and `learning_rate`.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X, y)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV F1 (weighted): {grid_search.best_score_:.4f}')

## 7. Confusion matrices for all models

Confusion matrices show where each classifier misclassifies: for example,
confusing Medium-risk communities with Low or High.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes_flat = axes.flat

class_names = list(label_encoder.classes_)

for ax, (name, model) in zip(axes_flat, trained_models.items()):
    # Use the same test set for consistency
    if name == 'Logistic Regression':
        y_pred = model.predict(scaler.transform(X_test))
    elif name == 'XGBoost':
        y_pred = model.predict(X_test_xgb)
        # use xgb test set
        cm = confusion_matrix(y_test_xgb, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
        disp.plot(ax=ax, cmap='Blues', colorbar=False)
        ax.set_title(name, fontsize=10)
        continue
    else:
        y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontsize=10)

# Hide unused subplot if any
for idx in range(len(trained_models), len(list(axes_flat))):
    axes_flat[idx].set_visible(False)

plt.suptitle('Confusion matrices for all classifiers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Save best model

In [ ]:
best_name = results_df['Accuracy'].astype(float).idxmax()
best_model = trained_models[best_name]
print(f'Best model: {best_name}')

save_model(best_model, scaler, label_encoder, feature_cols, model_dir='../models')
print('Model artifacts saved to ../models/')

## Summary

- Trained five classifiers (Logistic Regression, Decision Tree, Random Forest,
  Gradient Boosting, XGBoost) on community-level crime features.
- Five-fold cross-validation showed that ensemble methods (RF, GB, XGBoost)
  consistently outperform the single Decision Tree and Logistic Regression.
- Hyperparameter tuning via grid search improved the best model's F1 score.
- Confusion matrices reveal that Medium-risk communities are the hardest to
  classify correctly, which is expected since they sit at the boundary.
- The best model was saved for detailed evaluation in notebook 04.